# NeuroGolf 2026: EDA, Visualization

This notebook looks at the NeuroGolf task format, visualizes ARC-style grid transformations, and extracts simple statistics that are useful before building ONNX submissions.

**Main idea:** each task provides input-output grid examples. The submission is an ONNX network that reproduces the transformation exactly while keeping compute cost low.

## Problem Summary

NeuroGolf asks us to solve ARC-AGI image transformation tasks using the smallest possible neural networks.

- There are up to **400 tasks**, each submitted as one ONNX file: `task001.onnx`, `task002.onnx`, etc.
- Inputs are ARC grids: rectangular matrices with colors `0` to `9`.
- Each grid is converted before inference into a fixed one-hot tensor of shape `[1, 10, 30, 30]`.
- Outputs must also be `[1, 10, 30, 30]` one-hot tensors.
- A task only counts if the network is functionally correct on public examples, ARC-GEN examples, and private validation examples.

The challenge is not just solving transformations. It is solving them **cheaply**.

## Scoring and Constraints

For each correct task, the score is:

`max(1, 25 - ln(cost))`

where:

`cost = parameters + memory_bytes + MACs`

Important constraints:

- Static tensor shapes are required.
- ONNX file size must be at most **1.44 MB**.
- Disallowed ONNX ops: `Loop`, `Scan`, `NonZero`, `Unique`, `Script`, `Function`.
- Exact output matching is required; near-misses do not score.

From the silver/winning plans, a practical early strategy is to find easy tasks such as color remapping, geometric transforms, constant outputs, simple masking, and local convolution-like rules.

In [ ]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import ListedColormap, BoundaryNorm

KAGGLE_INPUT_DIR = Path("/kaggle/input/competitions/neurogolf-2026")
KAGGLE_UTILS_DIR = KAGGLE_INPUT_DIR / "neurogolf_utils"

sys.path.insert(0, str(KAGGLE_UTILS_DIR))
import neurogolf_utils

DATA_DIRS = [KAGGLE_INPUT_DIR]

print("Kaggle input directory:", KAGGLE_INPUT_DIR)
print("NeuroGolf utils:", KAGGLE_UTILS_DIR / "neurogolf_utils.py")


## Load Tasks

This loader works both locally, where we currently have `task001.json`, and on Kaggle, where the full competition input directory should contain more task JSON files.

In [ ]:
def find_task_file(task_num):
    name = f"task{task_num:03d}.json"
    for data_dir in DATA_DIRS:
        path = data_dir / name
        if path.exists():
            return path
    return None


def load_task(task_num):
    path = find_task_file(task_num)
    if path is None:
        raise FileNotFoundError(f"Could not find task{task_num:03d}.json")
    with path.open() as f:
        return json.load(f)


def available_task_numbers():
    nums = set()
    for data_dir in DATA_DIRS:
        if data_dir.exists():
            for path in data_dir.glob("task*.json"):
                stem = path.stem.replace("task", "")
                if stem.isdigit():
                    nums.add(int(stem))
    return sorted(nums)


task_nums = available_task_numbers()
print(f"Found {len(task_nums)} task JSON file(s):", task_nums[:20])

## ARC Color Palette

`neurogolf_utils` defines the official color palette and rendering utilities. The next cell shows the palette used throughout the notebook.

In [ ]:
neurogolf_utils.show_legend()

In [ ]:
ARC_COLORS = np.array(neurogolf_utils._COLORS[:10]) / 255.0
ARC_CMAP = ListedColormap(ARC_COLORS)
ARC_NORM = BoundaryNorm(np.arange(-0.5, 10.5, 1), ARC_CMAP.N)


def grid_array(grid):
    return np.array(grid, dtype=int)


def grid_shape(grid):
    arr = grid_array(grid)
    return arr.shape


def color_counts(grid):
    arr = grid_array(grid)
    values, counts = np.unique(arr, return_counts=True)
    return dict(zip(values.tolist(), counts.tolist()))


def plot_grid(grid, ax=None, title=None, show_values=False):
    arr = grid_array(grid)
    if ax is None:
        _, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(arr, cmap=ARC_CMAP, norm=ARC_NORM)
    ax.set_xticks(np.arange(-0.5, arr.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, arr.shape[0], 1), minor=True)
    ax.grid(which="minor", color="#222222", linewidth=0.7)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    if title:
        ax.set_title(title, fontsize=11)
    if show_values:
        for r in range(arr.shape[0]):
            for c in range(arr.shape[1]):
                text_color = "white" if arr[r, c] in [0, 9] else "black"
                ax.text(c, r, str(arr[r, c]), ha="center", va="center", color=text_color, fontsize=8)
    return ax


def plot_pairs(pairs, title="Examples", max_pairs=None, show_values=False):
    if max_pairs is not None:
        pairs = pairs[:max_pairs]
    n = len(pairs)
    fig, axes = plt.subplots(n, 2, figsize=(7, max(2.5, 2.7 * n)))
    if n == 1:
        axes = np.array([axes])
    for i, pair in enumerate(pairs):
        plot_grid(pair["input"], axes[i, 0], f"{title} {i}: input", show_values=show_values)
        plot_grid(pair["output"], axes[i, 1], f"{title} {i}: output", show_values=show_values)
    plt.tight_layout()
    return fig

## Dataset-Level Summary

These statistics describe dimensions, color usage, and subset sizes. With the full Kaggle data mounted, the same code summarizes all available tasks. In this local workspace it summarizes the files present here.

In [ ]:
def task_summary(task_num):
    task = load_task(task_num)
    rows = []
    for subset_name, pairs in task.items():
        for pair_idx, pair in enumerate(pairs):
            inp = grid_array(pair["input"])
            out = grid_array(pair["output"])
            rows.append({
                "task": task_num,
                "subset": subset_name,
                "pair": pair_idx,
                "input_h": inp.shape[0],
                "input_w": inp.shape[1],
                "output_h": out.shape[0],
                "output_w": out.shape[1],
                "same_shape": inp.shape == out.shape,
                "input_colors": tuple(sorted(np.unique(inp).tolist())),
                "output_colors": tuple(sorted(np.unique(out).tolist())),
                "n_input_colors": len(np.unique(inp)),
                "n_output_colors": len(np.unique(out)),
                "input_area": int(inp.size),
                "output_area": int(out.size),
                "area_ratio": out.size / inp.size,
            })
    return rows


summary_rows = []
for task_num in task_nums:
    summary_rows.extend(task_summary(task_num))

summary = pd.DataFrame(summary_rows)
summary

In [ ]:
if not summary.empty:
    display(summary.groupby("task").agg(
        pairs=("pair", "count"),
        subsets=("subset", lambda x: ", ".join(sorted(set(x)))),
        input_shapes=("input_h", lambda x: "see table"),
        same_shape_rate=("same_shape", "mean"),
        min_area_ratio=("area_ratio", "min"),
        max_area_ratio=("area_ratio", "max"),
    ))

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    summary["input_area"].hist(ax=axes[0], bins=20, color="#4c78a8")
    axes[0].set_title("Input area")
    summary["output_area"].hist(ax=axes[1], bins=20, color="#f58518")
    axes[1].set_title("Output area")
    summary["area_ratio"].hist(ax=axes[2], bins=20, color="#54a24b")
    axes[2].set_title("Output/Input area ratio")
    for ax in axes:
        ax.set_xlabel("cells or ratio")
        ax.set_ylabel("count")
    plt.tight_layout()

## Task 001: Visual Inspection

`task001.json` is available locally. It is a good first task because the input is a compact `3x3` grid and the output is a `9x9` grid. This strongly suggests a scaling or tiling rule.

In [ ]:
TASK_NUM = 1
task = load_task(TASK_NUM)
print({subset: len(pairs) for subset, pairs in task.items()})

for subset in ["train", "test", "arc-gen"]:
    print("\n", subset.upper())
    display(pd.DataFrame(task_summary(TASK_NUM)).query("subset == @subset"))

In [ ]:
plot_pairs(task["train"], title="Train", max_pairs=5, show_values=True);

In [ ]:
plot_pairs(task["test"], title="Test", show_values=True);

In [ ]:
plot_pairs(task["arc-gen"], title="ARC-GEN", max_pairs=6, show_values=False);

## Task 001 Hypothesis

For Task 001, the output appears to be a block expansion of the input. Each input cell controls a `3x3` block in the output:

- If the input cell is non-zero, copy the entire original `3x3` input pattern into the corresponding output block.
- If the input cell is zero, write a blank `3x3` block of zeros.

In compact NumPy form, this is similar to a Kronecker product between a non-zero mask and the input grid.

In [ ]:
def task001_rule(grid):
    arr = grid_array(grid)
    mask = (arr != 0).astype(int)
    return np.kron(mask, arr).astype(int)


def evaluate_python_rule(task, rule_fn):
    rows = []
    for subset, pairs in task.items():
        for i, pair in enumerate(pairs):
            pred = rule_fn(pair["input"])
            expected = grid_array(pair["output"])
            rows.append({
                "subset": subset,
                "pair": i,
                "correct": bool(np.array_equal(pred, expected)),
                "pred_shape": pred.shape,
                "expected_shape": expected.shape,
            })
    return pd.DataFrame(rows)


rule_eval = evaluate_python_rule(task, task001_rule)
display(rule_eval)
print("All examples solved by hypothesis:", rule_eval["correct"].all())

In [ ]:
example = task["train"][0]
pred = task001_rule(example["input"])

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
plot_grid(example["input"], axes[0], "Input", show_values=True)
plot_grid(example["output"], axes[1], "Expected", show_values=True)
plot_grid(pred, axes[2], "Hypothesis", show_values=True)
plt.tight_layout()

## One-Hot Tensor Format

`neurogolf_utils` provides `convert_to_numpy` and `convert_from_numpy`, matching the competition interface. This is useful for checking whether the grid-level analysis agrees with the tensor representation used by ONNX Runtime.

In [ ]:
sample = task["train"][0]
encoded = neurogolf_utils.convert_to_numpy(sample)

print("Encoded input shape:", encoded["input"].shape)
print("Encoded output shape:", encoded["output"].shape)
print("Decoded input matches original:", neurogolf_utils.convert_from_numpy(encoded["input"]) == sample["input"])
print("Decoded output matches original:", neurogolf_utils.convert_from_numpy(encoded["output"]) == sample["output"])

## Transformation Feature Checks

The cells below are useful when scaling from one task to many. They identify easy patterns mentioned in the silver and winning plans: same-shape transforms, color-only transforms, area scaling, constant outputs, and simple geometric relationships.

In [ ]:
def is_color_only_pair(pair):
    inp = grid_array(pair["input"])
    out = grid_array(pair["output"])
    if inp.shape != out.shape:
        return False
    mapping = {}
    for in_color, out_color in zip(inp.ravel(), out.ravel()):
        if in_color in mapping and mapping[in_color] != out_color:
            return False
        mapping[in_color] = out_color
    return True


def constant_outputs(task):
    outputs = []
    for subset in ["train", "test", "arc-gen"]:
        for pair in task.get(subset, []):
            outputs.append(grid_array(pair["output"]))
    if not outputs:
        return False
    first = outputs[0]
    return all(arr.shape == first.shape and np.array_equal(arr, first) for arr in outputs)


def geometric_matches(pair):
    inp = grid_array(pair["input"])
    out = grid_array(pair["output"])
    candidates = {
        "identity": inp,
        "flip_h": np.fliplr(inp),
        "flip_v": np.flipud(inp),
        "rot90": np.rot90(inp, 1),
        "rot180": np.rot90(inp, 2),
        "rot270": np.rot90(inp, 3),
        "transpose": inp.T,
    }
    return [name for name, arr in candidates.items() if arr.shape == out.shape and np.array_equal(arr, out)]


def classify_task(task):
    pairs = [pair for subset in task.values() for pair in subset]
    return {
        "all_same_shape": all(grid_shape(p["input"]) == grid_shape(p["output"]) for p in pairs),
        "constant_outputs": constant_outputs(task),
        "all_color_only_candidate": all(is_color_only_pair(p) for p in pairs),
        # "geometric_matches": [geometric_matches(p) for p in pairs],
        "area_ratios": sorted(set(grid_array(p["output"]).size / grid_array(p["input"]).size for p in pairs)),
    }


classify_task(task)

## Solver Takeaways From EDA

For this first local task:

- Input grids are `3x3`; output grids are `9x9`.
- The area ratio is `9`, pointing to a `3x3` block expansion.
- The Python hypothesis `np.kron(input != 0, input)` matches train, test, and ARC-GEN examples available locally.
- The ONNX version should express this rule with low-cost tensor operations rather than a large convolution.

For the full competition, the same checks can be run across the available task files and used to prioritize tasks with simple, low-cost rules.

In [ ]:
def score_from_cost(cost):
    return max(1.0, 25.0 - math.log(cost))


cost_examples = pd.DataFrame([
    {"network": "1x1 Conv 10->10", "params": 100, "memory": 400, "macs": 90_000},
    {"network": "3x3 Conv 10->10", "params": 900, "memory": 3_600, "macs": 810_000},
    {"network": "Two 1x1 Convs", "params": 200, "memory": 800, "macs": 180_000},
    {"network": "5x5 Conv 10->10", "params": 2_500, "memory": 10_000, "macs": 2_250_000},
])
cost_examples["cost"] = cost_examples[["params", "memory", "macs"]].sum(axis=1)
cost_examples["score"] = cost_examples["cost"].map(score_from_cost)
cost_examples

## Submission Notes

This notebook is only for EDA and rule inspection. For Task 001, the observed rule is `np.kron(input != 0, input)`, which matches the available examples in this notebook. The next notebook or script can focus on compiling that rule into ONNX and checking it with `neurogolf_utils.verify_network` before adding it to a submission zip.

The most useful tasks to target first are the ones that show consistent simple patterns: color remaps, flips/rotations, constant outputs, masking, scaling, or small local rules.